# Análise exploratória e XGBoost
Este notebook carrega os dados, cria um proxy de risco por sessão e treina um modelo XGBoost simples.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import xgboost as xgb

warnings.filterwarnings("ignore")

BASE_DIR = Path(r"c:\Users\josuc\OneDrive - PUCRS - BR\PUCRS\2026_01\TCC\AnaliseExploratória")
print("BASE_DIR exists:", BASE_DIR.exists())
print("CSV files found:", len(list(BASE_DIR.rglob("*_processed.csv"))))


BASE_DIR exists: True


CSV files found: 24


In [2]:
def load_all_csvs(base_dir):
    files = list(Path(base_dir).rglob("*_processed.csv"))
    datasets = {}
    for f in files:
        key = f.stem
        try:
            df = pd.read_csv(f)
            datasets[key] = df
        except Exception as e:
            print(f"Failed to read {f}: {e}")
    return datasets

def preprocess_df(df):
    df = df.copy()
    numeric_cols = ["missing_cbg","cbg","finger","basal","hr","gsr","carbInput","bolus"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "missing_cbg" in df.columns:
        df["missing_cbg"] = df["missing_cbg"].fillna(0)
    return df

def derive_proxy_label(df, cbg_threshold=250):
    high_cbg = False
    if "cbg" in df.columns and df["cbg"].notna().any():
        high_cbg = (df["cbg"] >= cbg_threshold).any()
    missing_ratio = df["missing_cbg"].mean() if "missing_cbg" in df.columns else 0.0
    high_risk = bool(high_cbg or (missing_ratio > 0.5))
    return 1 if high_risk else 0

def feature_engineer(df):
    df = df.copy()
    feats = {}
    for col in ["cbg","hr","gsr","carbInput","bolus","basal"]:
        if col in df.columns:
            feats[f"{col}_mean"] = float(df[col].mean(skipna=True)) if not df[col].dropna().empty else np.nan
            feats[f"{col}_std"] = float(df[col].std(skipna=True)) if not df[col].dropna().empty else np.nan
            feats[f"{col}_max"] = float(df[col].max(skipna=True)) if not df[col].dropna().empty else np.nan
            feats[f"{col}_min"] = float(df[col].min(skipna=True)) if not df[col].dropna().empty else np.nan
    if "missing_cbg" in df.columns:
        feats["missing_cbg_frac"] = float(df["missing_cbg"].mean())
    feats["n_rows"] = int(len(df))
    return feats

def prepare_dataset(base_dir):
    datasets = load_all_csvs(base_dir)
    rows = []
    for key, df in datasets.items():
        dfp = preprocess_df(df)
        feats = feature_engineer(dfp)
        label = derive_proxy_label(dfp)
        feats["session"] = key
        feats["label"] = label
        rows.append(feats)
    return pd.DataFrame(rows)

def train_xgb(df, label_col="label", random_state=42):
    df = df.copy()
    df = df.dropna(axis=1, how="all")
    if label_col not in df.columns:
        raise ValueError("label column missing")
    X = df.drop([label_col, "session"], axis=1, errors="ignore")
    y = df[label_col]
    stratify = y if len(y.unique()) > 1 else None
    X_train, X_test, y_train, y_test = train_test_split(X.fillna(0), y, test_size=0.25, stratify=stratify, random_state=random_state)
    model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=random_state)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    return model, X_train, X_test, y_train, y_test, {
        "roc_auc": roc_auc_score(y_test, y_proba) if y_proba is not None and len(np.unique(y_test)) > 1 else None,
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
    }

def evaluate_model(metrics):
    print("Evaluation metrics:")
    for key, value in metrics.items():
        print(f"{key}: {value}")


In [3]:
dataset = prepare_dataset(BASE_DIR)
print("Prepared dataset shape:", dataset.shape)
print(dataset.head())

assert not dataset.empty, "No data found in dataset."
assert "label" in dataset.columns, "Label column missing."

if dataset["label"].nunique() > 1 and len(dataset) >= 4:
    model, X_train, X_test, y_train, y_test, metrics = train_xgb(dataset)
    evaluate_model(metrics)
else:
    print("Not enough variability to train (need at least 2 label classes and 4 samples).")


Prepared dataset shape: (24, 28)
     cbg_mean    cbg_std  cbg_max  cbg_min    hr_mean     hr_std  hr_max  \
0  169.533015  66.958801    400.0     45.0  74.961246  17.545272   174.0   
1  167.372374  46.138496    313.0     62.0  97.458482  12.975930   162.0   
2  214.764299  66.396614    388.0     60.0  83.968003  10.563209   126.0   
3  150.493822  60.513732    342.0     40.0  78.576373  10.844037   133.0   
4  175.124328  48.680135    354.0     66.0  76.184741  14.776848   137.0   

   hr_min  gsr_mean   gsr_std  ...  bolus_max  bolus_min  basal_mean  \
0    51.0  0.629133  2.996022  ...        7.2        0.2    0.995772   
1    68.0  0.510253  1.965438  ...       17.1        1.1    0.860832   
2    64.0  0.089456  0.774886  ...       15.4        0.4    0.980594   
3    59.0  0.281014  1.351426  ...       11.8        0.8    0.721023   
4    52.0  0.103262  0.894895  ...        9.7        0.9    1.211523   

   basal_std  basal_max  basal_min  missing_cbg_frac  n_rows  \
0   0.316513 